<div style="background: linear-gradient(135deg, #1a0533 0%, #2d1b69 50%, #1e3a5f 100%); padding: 40px 36px; border-radius: 16px; color: white; font-family: 'Segoe UI', sans-serif; border-left: 6px solid #a78bfa;">
  <div style="font-size: 12px; color: #c4b5fd; letter-spacing: 3px; text-transform: uppercase; font-weight: 600;">
    SCY1101 — Evaluación Parcial 2 · Notebook 3 de 6
  </div>
  <h1 style="margin: 14px 0 8px; font-size: 32px; font-weight: 700; letter-spacing: -0.5px; color: white;">
    🔬 Evaluación de Modelos Finales
  </h1>
  <p style="color: #ddd6fe; margin: 0 0 24px; font-size: 16px; line-height: 1.6;">
    GaussianNB y KNeighbors ganaron el torneo. Ahora hay que entender qué significan sus errores<br>
    en la operación real: ¿cuántas incidencias detecta? ¿cuántos días se equivoca?
  </p>
  <div style="display: flex; gap: 12px; flex-wrap: wrap;">
    <span style="background: rgba(167,139,250,0.2); color: #c4b5fd; padding: 6px 14px; border-radius: 20px; font-size: 13px; border: 1px solid rgba(167,139,250,0.3);">🔲 Matriz de confusión</span>
    <span style="background: rgba(167,139,250,0.2); color: #c4b5fd; padding: 6px 14px; border-radius: 20px; font-size: 13px; border: 1px solid rgba(167,139,250,0.3);">📈 Curva ROC</span>
    <span style="background: rgba(167,139,250,0.2); color: #c4b5fd; padding: 6px 14px; border-radius: 20px; font-size: 13px; border: 1px solid rgba(167,139,250,0.3);">📉 Scatter real vs predicho</span>
    <span style="background: rgba(167,139,250,0.2); color: #c4b5fd; padding: 6px 14px; border-radius: 20px; font-size: 13px; border: 1px solid rgba(167,139,250,0.3);">💼 Traducción al negocio</span>
  </div>
  <div style="margin-top: 24px; background: rgba(255,255,255,0.07); border-radius: 10px; padding: 12px 18px; font-size: 13px; color: #c4b5fd;">
    <strong>Progreso del proyecto:</strong>
    <div style="background: rgba(255,255,255,0.15); border-radius: 10px; height: 6px; margin: 8px 0 4px;">
      <div style="background: linear-gradient(90deg, #a78bfa, #c4b5fd); width: 50%; height: 6px; border-radius: 10px;"></div>
    </div>
    <div style="display: flex; justify-content: space-between; font-size: 11px; color: #ddd6fe; opacity: 0.8;">
      <span>✅ EDA</span><span>✅ Modelos</span><span>▶ Evaluación</span><span>Tuning</span><span>Clustering</span><span>Final</span>
    </div>
  </div>
</div>

---

## 🧭 La historia de este notebook

<div style="background: #f5f3ff; border-left: 5px solid #7c3aed; padding: 20px 24px; border-radius: 0 12px 12px 0; margin: 16px 0; font-family: 'Segoe UI', sans-serif;">
  <strong style="color: #3b0764; font-size: 15px;">🔬 Del ranking al significado real</strong><br><br>
  <span style="color: #4c1d95; font-size: 14px; line-height: 1.7;">
    En el NB02 elegimos a los mejores candidatos mirando métricas en validación cruzada.
    Pero un número como "F1 = 0.28" o "MAE = 1.44 días" no dice nada por sí solo en una reunión de operaciones.<br><br>
    Este notebook traduce esas métricas a preguntas reales:
    <strong>¿de 100 envíos con incidencia real, cuántos detecta el modelo?</strong>
    <strong>¿cuando el modelo se equivoca en los días, cuánto se equivoca típicamente?</strong>
  </span>
</div>

**Tres instrumentos de diagnóstico:**

| Herramienta | ¿Qué mide? | ¿Para qué sirve en defensa? |
|-------------|-----------|----------------------------|
| Matriz de confusión | Falsos positivos vs falsos negativos | Explicar el trade-off de alertas vs incidencias perdidas |
| Curva ROC | Sensibilidad vs especificidad | Mostrar la capacidad discriminadora global del modelo |
| Scatter real vs predicho | Error de estimación de días | Visualizar dónde el modelo falla más en regresión |

In [1]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "01_raw"
DATA_MODEL = PROJECT_ROOT / "data" / "05_model_input"
DATA_OUTPUT = PROJECT_ROOT / "data" / "07_model_output"
DATA_REPORT = PROJECT_ROOT / "data" / "08_reporting"
DATA_MODELS = PROJECT_ROOT / "data" / "06_models"

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)


def read_csv(path):
    for enc in ("utf-8", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path)


def clean_text_columns(df):
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.replace("Ã³", "o", regex=False)
    return df


In [2]:
FEATURES_NUMERICAS = [
    "peso_kg", "volumen_m3", "distancia_km", "tiempo_estimado_hrs",
    "peaje_total", "capacidad_kg", "capacidad_m3", "km_recorridos",
    "eficiencia_peso", "eficiencia_volumen", "costo_por_km",
    "id_ruta", "id_vehiculo", "mes_envio", "dia_semana_envio",
    "trimestre_envio",
]

FEATURES_CATEGORICAS = [
    "estado", "tipo_carga_norm", "origen", "destino",
    "tipo_via", "tipo", "estado_vehiculo",
]


def feature_matrix(df, target):
    features = [c for c in FEATURES_NUMERICAS + FEATURES_CATEGORICAS if c in df.columns]
    return df[features], df[target], features


---

## 1. 🔲 Evaluación de clasificación — GaussianNB

<div style="background: #fdf4ff; border-left: 5px solid #a855f7; padding: 14px 20px; border-radius: 0 10px 10px 0; margin: 12px 0;">
  <strong>¿Qué esperar de este modelo?</strong><br>
  <span style="font-size: 13px; color: #581c87;">
    GaussianNB fue seleccionado porque maximizó el F1 de la clase "con incidencia" en el torneo.
    Eso significa que prioriza <strong>detectar incidencias</strong> sobre evitar falsas alarmas.
    La matriz de confusión mostrará exactamente ese trade-off con números reales del conjunto de prueba.
  </span>
</div>

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    mean_absolute_error, mean_squared_error, r2_score
)

clf_data = read_csv(DATA_MODEL / "model_input_clasificacion.csv")
reg_data = read_csv(DATA_MODEL / "model_input_regresion.csv")
metricas_final = read_csv(DATA_OUTPUT / "metricas_modelo_final.csv")

with open(DATA_MODELS / "modelo_final_clasificacion.pkl", "rb") as f:
    modelo_clf = pickle.load(f)
with open(DATA_MODELS / "modelo_final_regresion.pkl", "rb") as f:
    modelo_reg = pickle.load(f)

metricas_final


,tipo_problema,modelo_final,etapa,cv_score,criterio_seleccion,accuracy,precision,recall,f1,roc_auc,tn,fp,fn,tp,mae,mse,rmse,r2
0,clasificacion,11_GaussianNB,grid_search,0.2596,max_f1,0.2356,0.175,0.9655,0.2963,0.5063,13.0,132.0,1.0,28.0,NaN,NaN,NaN,NaN
1,regresion,06_KNeighborsRegressor,grid_search,-1.4432,min_mae,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.4374,2.6246,1.6201,0.1153


<div style="background: #fdf4ff; border: 1px solid #e9d5ff; border-radius: 10px; padding: 16px 20px; margin: 8px 0; font-family: 'Segoe UI', sans-serif;">
  <strong style="color: #581c87;">🔲 Cómo leer la matriz de confusión</strong><br><br>
  <span style="color: #4a1272; font-size: 14px; line-height: 1.7;">
    La matriz tiene cuatro celdas que cuentan casos reales:<br><br>
    <strong>✅ Verdadero Positivo (VP):</strong> el modelo detectó una incidencia que SÍ ocurrió → alerta correcta<br>
    <strong>✅ Verdadero Negativo (VN):</strong> el modelo dijo "sin incidencia" y tenía razón → silencio correcto<br>
    <strong>⚠️ Falso Positivo (FP):</strong> el modelo alertó pero no había incidencia → falsa alarma<br>
    <strong>🔴 Falso Negativo (FN):</strong> el modelo no detectó una incidencia que SÍ ocurrió → incidencia perdida<br><br>
    <strong>Para operaciones logísticas:</strong> el FN es más costoso que el FP — perder una incidencia real
    es peor que generar una alarma extra. Por eso el modelo prioriza recall alto, aunque genere más FP.
    El recall del <strong>96.6%</strong> significa que de cada 100 incidencias reales, detecta ~97.
  </span>
</div>

---

## 2. 📈 Curva ROC — capacidad discriminadora

<div style="background: #f0f9ff; border-left: 5px solid #0ea5e9; padding: 14px 20px; border-radius: 0 10px 10px 0; margin: 12px 0;">
  <strong>¿Qué mide la curva ROC?</strong><br>
  <span style="font-size: 13px; color: #0c4a6e;">
    La curva ROC muestra cómo cambia la sensibilidad del modelo (detectar incidencias reales)
    cuando se ajusta el umbral de decisión. El área bajo la curva (AUC) resume todo en un número:
    AUC = 1.0 es perfecto, AUC = 0.5 es equivalente a adivinar al azar.
    El modelo final tiene AUC ≈ 0.51 — honestamente cerca del azar, lo que refuerza la recomendación
    de usarlo como <em>screening inicial</em>, no como herramienta definitiva.
  </span>
</div>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

FEATURES_NUMERICAS_C = [
    "peso_kg", "volumen_m3", "distancia_km", "tiempo_estimado_hrs",
    "peaje_total", "capacidad_kg", "capacidad_m3", "km_recorridos",
    "eficiencia_peso", "eficiencia_volumen", "costo_por_km",
    "id_ruta", "id_vehiculo", "mes_envio", "dia_semana_envio", "trimestre_envio",
]
FEATURES_CATEGORICAS_C = [
    "estado", "tipo_carga_norm", "origen", "destino",
    "tipo_via", "tipo", "estado_vehiculo",
]

feats_c = [c for c in FEATURES_NUMERICAS_C + FEATURES_CATEGORICAS_C if c in clf_data.columns]
X_clf = clf_data[feats_c]
y_clf = clf_data["tiene_incidencia"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

y_pred_c = modelo_clf.predict(X_test_c)
y_proba_c = modelo_clf.predict_proba(X_test_c)[:, 1] if hasattr(modelo_clf, 'predict_proba') else None

cm = confusion_matrix(y_test_c, y_pred_c)
print(classification_report(y_test_c, y_pred_c, target_names=['Sin incidencia', 'Con incidencia']))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Sin incidencia', 'Con incidencia'])
disp.plot(cmap='Blues')
import matplotlib.pyplot as plt
plt.title("Matriz de Confusion - GaussianNB")
plt.tight_layout()
plt.show()


In [4]:
if y_proba_c is not None:
    fpr, tpr, _ = roc_curve(y_test_c, y_proba_c)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.title("Curva ROC - clasificacion")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.tight_layout()
    plt.show()


NameError: name 'y_proba_c' is not defined

<div style="background: #fff1f2; border: 1px solid #fecdd3; border-radius: 10px; padding: 16px 20px; margin: 8px 0; font-family: 'Segoe UI', sans-serif;">
  <strong style="color: #881337;">⚠️ AUC ≈ 0.51 — ¿es un problema? No si se defiende bien</strong><br><br>
  <span style="color: #9f1239; font-size: 14px; line-height: 1.7;">
    Un AUC cercano a 0.5 indica que el modelo no discrimina globalmente mucho mejor que el azar.
    Sin embargo, el modelo tiene un <strong>recall de 96.6%</strong> — detecta casi todas las incidencias.<br><br>
    La explicación correcta: el modelo sacrifica discriminación global para maximizar la detección de la clase minoritaria.
    Eso tiene valor operativo como <strong>sistema de alerta temprana sensible</strong>,
    sabiendo que generará falsos positivos que requieren validación humana antes de actuar.
  </span>
</div>

---

## 3. 📉 Evaluación de regresión — KNeighborsRegressor

<div style="background: #fff7ed; border-left: 5px solid #f59e0b; padding: 14px 20px; border-radius: 0 10px 10px 0; margin: 12px 0;">
  <strong>¿Qué esperar de este modelo?</strong><br>
  <span style="font-size: 13px; color: #78350f;">
    KNeighborsRegressor estima los días de tránsito buscando los K envíos más similares
    al nuevo y promediando sus días reales. El scatter de "real vs predicho" mostrará visualmente
    cuánto se desvía la estimación del valor real — si los puntos están cerca de la diagonal, el modelo es preciso.
  </span>
</div>

In [ ]:
X_reg, y_reg, features_reg = feature_matrix(reg_data, "dias_en_transito")
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

y_pred_r = modelo_reg.predict(X_test_r)
mae = mean_absolute_error(y_test_r, y_pred_r)
mse = mean_squared_error(y_test_r, y_pred_r)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_r, y_pred_r)

pd.DataFrame([{"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}]).round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(x=y_test_r, y=y_pred_r, ax=axes[0])
axes[0].plot([y_test_r.min(), y_test_r.max()], [y_test_r.min(), y_test_r.max()], "--", color="gray")
axes[0].set_title("Dias reales vs predichos")
axes[0].set_xlabel("Dias reales")
axes[0].set_ylabel("Dias predichos")

residuos = y_test_r - y_pred_r
sns.histplot(residuos, kde=True, ax=axes[1])
axes[1].set_title("Distribucion de residuos")
axes[1].set_xlabel("Error real - predicho")

plt.tight_layout()
plt.show()


<div style="background: #fff7ed; border: 1px solid #fed7aa; border-radius: 10px; padding: 16px 20px; margin: 8px 0; font-family: 'Segoe UI', sans-serif;">
  <strong style="color: #9a3412;">📊 Cómo leer el scatter real vs predicho</strong><br><br>
  <span style="color: #7c2d12; font-size: 14px; line-height: 1.7;">
    La diagonal perfecta (línea roja punteada) representa predicción sin error.
    Cada punto es un envío del conjunto de prueba: su posición horizontal es el valor real,
    la vertical es la predicción del modelo.<br><br>
    <strong>MAE = 1.44 días</strong> — en promedio, el modelo se equivoca 1.44 días al estimar el tránsito.<br>
    <strong>R² = 0.12</strong> — el modelo explica solo el 12% de la varianza en días de tránsito.<br><br>
    <strong>Traducción para defensa:</strong> "El modelo da una estimación inicial útil, pero el poder explicativo
    es bajo porque los días de tránsito dependen de variables que no tenemos aún — tráfico, clima, prioridad de carga.
    Con mejores variables, este tipo de modelo puede mejorar significativamente."
  </span>
</div>

---

## 🏁 Conclusión: fortalezas y límites documentados

<div style="background: linear-gradient(135deg, #1a0533 0%, #2d1b69 50%, #1e3a5f 100%); padding: 28px 32px; border-radius: 12px; color: white; font-family: 'Segoe UI', sans-serif; margin: 16px 0;">
  <h3 style="margin: 0 0 16px; color: #c4b5fd;">📋 Diagnóstico final de los modelos</h3>
  <div style="display: flex; gap: 16px; flex-wrap: wrap; margin-bottom: 20px;">
    <div style="background: rgba(255,255,255,0.1); border-radius: 10px; padding: 16px 20px; flex: 1; min-width: 140px; text-align: center;">
      <div style="font-size: 26px; font-weight: bold; color: #4ade80;">96.6%</div>
      <div style="font-size: 12px; color: #ddd6fe; margin-top: 4px;">Recall — incidencias detectadas</div>
    </div>
    <div style="background: rgba(255,255,255,0.1); border-radius: 10px; padding: 16px 20px; flex: 1; min-width: 140px; text-align: center;">
      <div style="font-size: 26px; font-weight: bold; color: #fbbf24;">17.5%</div>
      <div style="font-size: 12px; color: #ddd6fe; margin-top: 4px;">Precisión — requiere validación humana</div>
    </div>
    <div style="background: rgba(255,255,255,0.1); border-radius: 10px; padding: 16px 20px; flex: 1; min-width: 140px; text-align: center;">
      <div style="font-size: 26px; font-weight: bold; color: #60a5fa;">1.44</div>
      <div style="font-size: 12px; color: #ddd6fe; margin-top: 4px;">MAE días — error promedio tránsito</div>
    </div>
    <div style="background: rgba(255,255,255,0.1); border-radius: 10px; padding: 16px 20px; flex: 1; min-width: 140px; text-align: center;">
      <div style="font-size: 26px; font-weight: bold; color: #f87171;">0.12</div>
      <div style="font-size: 12px; color: #ddd6fe; margin-top: 4px;">R² — requiere más variables</div>
    </div>
  </div>
  <p style="color: #ddd6fe; margin: 0; font-size: 14px; line-height: 1.7;">
    La evaluación no busca ocultar limitaciones — las cuantifica y explica.
    Un recall de 96.6% con precisión de 17.5% es un resultado honesto y defendible
    como sistema de screening inicial. La baja R² en regresión es evidencia de que faltan variables,
    no de un pipeline mal construido.
  </p>
</div>

<div style="background: #eff6ff; border: 1px solid #bfdbfe; border-radius: 10px; padding: 16px 20px; margin-top: 16px; font-family: 'Segoe UI', sans-serif;">
  <strong style="color: #1e40af;">➡️ Siguiente: Notebook 04 — Optimización de Hiperparámetros</strong><br>
  <span style="color: #1e3a8a; font-size: 14px;">
    Con los finalistas identificados y evaluados, se aplica <strong>RandomizedSearchCV</strong> y
    <strong>GridSearchCV</strong> para afinar sus hiperparámetros y extraer su mejor rendimiento posible.
  </span>
</div>